# Linear Regression

**Linear regression** is a **supervised learning** algorithm used for **predicting a continuous dependent variable** based on one or more independent variables (features). The goal is to find a **linear relationship** between the inputs and the output.

At its core, it tries to fit a **line (or hyperplane)** that best predicts the target variable.

<br>

<p align="center">
<img src="visualizations/linear_regression.png" width="600">
</p>

### Mathematical Model

For **simple linear regression** (one feature), the model is:

$$
\hat{y} = w x + b
$$

Where:

* $\hat{y}$: predicted output
* $x$: input feature
* $w$: weight (slope)
* $b$: bias (intercept)

In **multiple linear regression** (multiple features), it's:

$$
\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w}^\top \mathbf{x} + b
$$

This is a **parametric model**, meaning it makes strong assumptions about the form of the data.

---

### Objective Function

The objective is to minimize the **error** between predicted values $\hat{y}$ and the true values $y$. The most common error metric is the **Mean Squared Error (MSE)**:

$$
\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$$

This is a **convex** function, which guarantees that gradient-based methods can converge to a global minimum.

---

### Optimization

To find the optimal $\mathbf{w}, b$, you can use:

#### 1. **Ordinary Least Squares (OLS)**:

* Closed-form solution using linear algebra:

$$
\mathbf{w} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}
$$

This is computationally efficient for small/moderate-sized data but expensive for large-scale data due to matrix inversion.

#### 2. **Gradient Descent**:

* Iterative optimization technique:

$$
\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_\mathbf{w} \text{MSE}
$$

Where $\eta$ is the learning rate. More scalable for high-dimensional or big data problems.

---

### Assumptions

Linear regression assumes:

1. **Linearity**: Relationship between features and target is linear.
2. **Independence**: Observations are independent.
3. **Homoscedasticity**: Constant variance of residuals.
4. **No multicollinearity**: Features are not highly correlated.
5. **Normality of residuals** (for inference, not prediction).

Violation of these assumptions affects model validity.

---


### Beyond Linearity

**Basis Expansion (e.g. polynomial, splines, RBF features):**
Explicitly transforms inputs using nonlinear functions like $x^2, x^3$, splines, or Gaussian bumps.
It basically **bends the hyperplane**.

**Kernel Methods (e.g. SVM, Kernel Ridge):**
Use kernel functions (e.g. RBF, polynomial) to compute dot products in an implicit high-dimensional space.
It **imagines the data in a curved space**.

In [5]:
import numpy as np
import pandas as pd
from numpy.linalg import inv

In [ ]:
data = pd.read_csv('california_housing.csv')
data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [19]:
numeric_data = data.select_dtypes(include=[np.number])
numeric_data = numeric_data.dropna()
numeric_data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0
...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0


In [76]:
# Split features and target
target = "median_house_value"
X = numeric_data.drop(columns=target).values
y = numeric_data[target].values.reshape(-1, 1)

# Standardize features for better numerical stability (mean 0, std 1)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_norm = (X - X_mean) / X_std

# Add bias term (intercept)
X_bias = np.hstack([np.ones((X_norm.shape[0], 1)), X_norm])

# Train/test split
split_idx = int(0.8 * len(X_bias))
X_train, X_test = X_bias[:split_idx], X_bias[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Closed-form linear regression: w = (XᵀX)⁻¹ Xᵀy
w = inv(X_train.T @ X_train) @ X_train.T @ y_train

# Predictions
y_pred = X_test @ w

# Mean Squared Error
mse = np.mean((y_test - y_pred) ** 2)
print("MSE:", mse)
print("Percentual error: ", np.sqrt(mse)/y.mean())

MSE: 4471584111.393299
Percentual error:  0.32325475105698953


# Regularization
By selecting and tuning the right regularizer, you can improve predictive performance, interpretability, and numerical stability of your linear regression models.


## 1. Ridge Regression (L₂ Regularization)

### Formulation

Ridge regression adds an L₂ penalty (squared 2-norm) on the coefficients:

$$
\min_{\mathbf w}\;\sum_{i=1}^n\bigl(y_i - \mathbf w^T \mathbf x_i\bigr)^2 \;+\;\lambda\,\|\mathbf w\|_2^2
\quad\text{where}\quad
\|\mathbf w\|_2^2 = \sum_{j=1}^p w_j^2.
$$

* **λ (lambda)** ≥ 0 is a hyperparameter controlling the strength of regularization.

  * λ = 0 → equivalent to OLS.
  * λ → ∞ → shrinks all weights toward zero.

### Intuition

* The penalty term $\lambda \sum_j w_j^2$ discourages any single coefficient from growing too large.
* Helps when features are **collinear** or when $p$ (number of features) is comparable to or larger than $n$ (number of samples).
* Results in a **closed-form solution**:

  $$
    \hat{\mathbf w}_{\text{ridge}}
    = \bigl(\mathbf X^T \mathbf X + \lambda \mathbf I\bigr)^{-1}\,\mathbf X^T \mathbf y,
  $$

  which is always invertible (even if $\mathbf X^T \mathbf X$ is singular).

### Effects on Coefficients

* **Shrinks** all coefficients continuously toward zero, but typically **none** become exactly zero.
* Useful for **stability** and when you believe most features have some (possibly small) predictive power.

### Geometric Perspective

* OLS finds the point where the error contours (ellipses) intersect the weight axes.
* Ridge places a **circular** (L₂) constraint region; the optimum is where the smallest ellipse touches this circle.
* Because the region is round, the solution tends to keep all components nonzero but small.

---

## 2. Lasso Regression (L₁ Regularization)

### Formulation

Lasso adds an L₁ penalty (sum of absolute values) on the coefficients:

$$
\min_{\mathbf w}\;\sum_{i=1}^n\bigl(y_i - \mathbf w^T \mathbf x_i\bigr)^2 \;+\;\lambda\,\|\mathbf w\|_1
\quad\text{where}\quad
\|\mathbf w\|_1 = \sum_{j=1}^p |w_j|.
$$

* Again, λ ≥ 0 controls regularization strength.

### Intuition

* The L₁ penalty $\lambda \sum_j |w_j|$ encourages sparsity: it’s “cheaper” (in penalty terms) to set some coefficients exactly to zero.
* Acts as both **feature selection** (zeroing out irrelevant weights) and coefficient shrinkage.
* There is **no closed-form**; typically solved via coordinate descent or LARS.

### Effects on Coefficients

* Drives many coefficients exactly to **zero** when λ is sufficiently large.
* Yields a **sparse** model that’s easier to interpret (only a subset of features remain).

### Geometric Perspective

* Lasso uses a **diamond-shaped** (L₁) constraint region; the sharp corners align with axes.
* The smallest error ellipse often touches at a corner → many weights exactly zero.

---

## 3. Choosing Between Ridge and Lasso

| Aspect                   | Ridge (L₂)                       | Lasso (L₁)            |
| ------------------------ | -------------------------------- | --------------------- |
| Penalty                  | $\sum w_j^2$                     | $\sum w_j$            |
| Solution path            | Smooth shrinkage                 | Sparse selection      |
| Closed-form solution     | Yes                              | No                    |
| Coefficients driven to 0 | No                               | Yes (for large λ)     |
| Best when                | Many small effects; collinearity | Few strong predictors |

* **If you believe** that *many* features have small but nonzero effects, **Ridge** often wins.
* **If you believe** that *only a handful* of features drive the outcome, **Lasso** can find those key predictors.


### Key Takeaways

1. **Regularization** adds bias to reduce variance and overfitting.
2. **Ridge (L₂)** gently shrinks all weights—best when you expect diffuse influences.
3. **Lasso (L₁)** can zero out weights—best when you want a simple, interpretable model.
4. **Hyperparameter λ** must be chosen (e.g., via cross-validation) to balance bias and variance optimally.

In [75]:
# Split features and target
target = "median_house_value"
X = numeric_data.drop(columns=target).values
y = numeric_data[target].values.reshape(-1, 1)

# Standardize features
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0)
X_norm = (X - X_mean) / X_std

# Add bias term
X_bias = np.hstack([np.ones((X_norm.shape[0], 1)), X_norm])

# Train/test split
split_idx    = int(0.8 * len(X_bias))
X_train, X_test = X_bias[:split_idx], X_bias[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# ——— Ridge closed-form ———
lambda_ = 10.0   # regularization strength

# Build identity (do not penalize bias term at index 0)
I = np.eye(X_train.shape[1])
I[0,0] = 0

# Closed-form ridge solution
w_ridge = inv(X_train.T @ X_train + lambda_ * I) @ (X_train.T @ y_train)

# Predictions & eval
y_pred = X_test @ w_ridge
mse_ridge = np.mean((y_test - y_pred)**2)
print("Ridge MSE:", mse_ridge)
print("Percentual error: ", np.sqrt(mse_ridge)/y.mean())

Ridge MSE: 4471417880.75597
Percentual error:  0.32324874252160757


In [80]:
# Split features and target
target = "median_house_value"
X = numeric_data.drop(columns=target).values
y = numeric_data[target].values.reshape(-1, 1)

# Standardize features
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0)
X_norm = (X - X_mean) / X_std

# Add bias term
X_bias = np.hstack([np.ones((X_norm.shape[0], 1)), X_norm])

# Train/test split
split_idx    = int(0.8 * len(X_bias))
X_train, X_test = X_bias[:split_idx], X_bias[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# ——— Lasso via (sub)gradient descent ———
lambda_   = 0.01    # L1 strength
alpha     = 0.1  # learning rate
n_iters   = 10000

# Initialize weights
w_lasso = np.zeros((X_train.shape[1], 1))

for i in range(n_iters):
    # Prediction error
    y_hat = X_train @ w_lasso
    error = y_hat - y_train
    
    # Gradient of MSE part
    grad = 2 * (X_train.T @ error) / len(y_train)
    
    # Subgradient of L1: sign(w), except zero → 0
    subgrad = np.sign(w_lasso)
    
    # Don’t regularize bias term
    subgrad[0] = 0

    # Full gradient step
    w_lasso -= alpha * (grad + lambda_ * subgrad)

# Evaluate
y_pred_lasso = X_test @ w_lasso
mse_lasso    = np.mean((y_test - y_pred_lasso)**2)
print("Lasso MSE:", mse_lasso)
print("Percentual error: ", np.sqrt(mse_lasso)/y.mean())

Lasso MSE: 4471583884.501441
Percentual error:  0.3232547428558834
